# Interactive 3D Gaze + Head Scene Viewer

这个 notebook 用 `plotly` 在 Scene frame 里交互式显示：

- 黑色：head/gaze origin trajectory（CPF origin）
- 蓝色：head forward direction
- 红色：gaze direction

特点：

- 鼠标默认只做 zoom
- 视角旋转改成受控的 `azimuth` 滑块
- 可以指定哪个 Scene 轴当作现实里的“上”
- 不再依赖自由 3D 拖拽

使用前提：

1. 已经有 `*_gaze_samples.csv`
2. 已经有 `*_head_samples.csv`
3. notebook 运行在 `adt` 环境里


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import ipywidgets as widgets
from IPython.display import display, clear_output

from adt_sandbox.head_viewer import (
    discover_sequence_ids,
    load_gaze_head_frame,
    plot_gaze_head_scene_window_plotly,
    slice_frame_window,
)


In [2]:
REPORTS_DIR = Path('/mnt/d/SparseGaze/ADT-Gaze')
SEQUENCE_IDS = discover_sequence_ids(REPORTS_DIR)
CACHE = {}

def get_frame(sequence_id: str):
    if sequence_id not in CACHE:
        CACHE[sequence_id] = load_gaze_head_frame(REPORTS_DIR, sequence_id)
    return CACHE[sequence_id]

print(f'sequences: {len(SEQUENCE_IDS)}')
SEQUENCE_IDS[:5]


sequences: 34


['Apartment_release_decoration_skeleton_seq131_M1292',
 'Apartment_release_decoration_skeleton_seq132_M1292',
 'Apartment_release_decoration_skeleton_seq133_M1292',
 'Apartment_release_decoration_skeleton_seq134_M1292',
 'Apartment_release_decoration_skeleton_seq135_M1292']

In [3]:
sequence_dropdown = widgets.Dropdown(
    options=SEQUENCE_IDS,
    value=SEQUENCE_IDS[0],
    description='sequence',
    layout=widgets.Layout(width='680px'),
)

start_input = widgets.BoundedIntText(value=0, min=0, max=100, step=1, description='start')
end_input = widgets.BoundedIntText(value=120, min=1, max=120, step=1, description='end')
stride_input = widgets.BoundedIntText(value=10, min=1, max=60, step=1, description='stride')
head_scale_slider = widgets.FloatSlider(value=0.35, min=0.05, max=1.5, step=0.05, description='head_scale', continuous_update=False)
gaze_scale_mode = widgets.Dropdown(options=['fixed', 'depth'], value='fixed', description='gaze_scale')
gaze_fixed_scale_slider = widgets.FloatSlider(value=0.35, min=0.05, max=1.5, step=0.05, description='gaze_len', continuous_update=False)
vertical_axis_dropdown = widgets.Dropdown(
    options=[('Scene Y up', 'scene_y'), ('Scene Z up', 'scene_z'), ('Scene X up', 'scene_x')],
    value='scene_y',
    description='vertical',
)
azimuth_slider = widgets.FloatSlider(value=35.0, min=0.0, max=360.0, step=5.0, description='azimuth', continuous_update=False)
show_trajectory_checkbox = widgets.Checkbox(value=True, description='show trajectory')
render_button = widgets.Button(description='Render', button_style='primary', icon='refresh')
output = widgets.Output()

def update_bounds(*_):
    frame = get_frame(sequence_dropdown.value)
    max_row = max(len(frame) - 1, 1)
    start_input.max = max_row
    end_input.max = len(frame)
    if start_input.value > max_row:
        start_input.value = 0
    if end_input.value <= start_input.value:
        end_input.value = min(len(frame), start_input.value + 120)

def render(*_):
    frame = get_frame(sequence_dropdown.value)
    end_value = max(end_input.value, start_input.value + 1)
    window = slice_frame_window(
        frame,
        start_row=start_input.value,
        end_row=end_value,
        stride=stride_input.value,
    )
    with output:
        clear_output(wait=True)
        fig = plot_gaze_head_scene_window_plotly(
            window,
            head_scale_m=head_scale_slider.value,
            gaze_scale_mode=gaze_scale_mode.value,
            fixed_gaze_scale_m=gaze_fixed_scale_slider.value,
            show_trajectory=show_trajectory_checkbox.value,
            vertical_axis=vertical_axis_dropdown.value,
            azimuth_deg=azimuth_slider.value,
        )
        display(fig)

def update_sequence(*_):
    update_bounds()
    render()

sequence_dropdown.observe(update_sequence, names='value')
render_button.on_click(render)

update_bounds()
render()

display(
    widgets.VBox([
        sequence_dropdown,
        widgets.HBox([start_input, end_input, stride_input]),
        widgets.HBox([head_scale_slider, gaze_scale_mode, gaze_fixed_scale_slider]),
        widgets.HBox([vertical_axis_dropdown, azimuth_slider]),
        show_trajectory_checkbox,
        render_button,
        output,
    ])
)
